In [ ]:
!git clone https://github.com/facefusion/facefusion.git
%cd /content/facefusion
!pip install -r requirements.txt
!pip install onnxruntime-gpu

In [ ]:
# Path to the core.py file
file_path = '/content/facefusion/facefusion/uis/layouts/default.py'

# Read the content of the file
with open(file_path, 'r') as file:
    content = file.read()

# Replace the line with the updated version
updated_content = content.replace("ui.launch(show_api = False)", "ui.launch(show_api = False, share=True, enable_queue=True)")

# Write the updated content back to the file
with open(file_path, 'w') as file:
    file.write(updated_content)


In [ ]:
import os, subprocess, time, re
from google.colab import drive

# 1. 挂载 Google Drive 并在云盘创建模型缓存目录
drive.mount('/content/drive')
drive_models = "/content/drive/MyDrive/facefusion_models"
os.makedirs(drive_models, exist_ok=True)

# 2. 自动检查并安装 FaceFusion
if not os.path.exists("/content/facefusion"):
    !git clone https://github.com/facefusion/facefusion.git /content/facefusion
    %cd /content/facefusion
    !python install.py cuda@12 --skip-conda

# 3. 将 FaceFusion 的模型目录软链接到云盘（核心防重下步骤）
local_models = "/content/facefusion/.assets/models"
os.makedirs("/content/facefusion/.assets", exist_ok=True)
if os.path.exists(local_models) and not os.path.islink(local_models):
    !rm -rf {local_models}
if not os.path.islink(local_models):
    os.symlink(drive_models, local_models)

# 4. 启动 Cloudflare 隧道并获取链接
if subprocess.run(["which", "cloudflared"], capture_output=True).returncode != 0:
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

with open("/content/cf.log", "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"], stdout=f, stderr=f)

url = None
for _ in range(30):
    time.sleep(1)
    with open("/content/cf.log") as f:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
        if match:
            url = match.group(0)
            break

print("👉 公网访问链接:", url)

# 5. 启动 FaceFusion
%cd /content/facefusion
!python facefusion.py run --execution-providers cuda

ValueError: mount failed

In [ ]:
!python run.py